In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator

print("1. Εκκίνηση SparkSession...")
spark = SparkSession.builder \
    .appName("RF_SMOTE_GridSearch") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

print("2. Φόρτωση και Feature Augmentation...")
train_gold = spark.read.parquet("../../data/train_gold_with_cluster.parquet")
test_gold = spark.read.parquet("../../data/test_gold_with_cluster.parquet")

train_gold = train_gold.withColumn("stroke", train_gold["stroke"].cast(DoubleType()))
test_gold = test_gold.withColumn("stroke", test_gold["stroke"].cast(DoubleType()))
train_gold = train_gold.withColumn("cluster", F.col("cluster").cast(DoubleType()))
test_gold = test_gold.withColumn("cluster", F.col("cluster").cast(DoubleType()))

assembler = VectorAssembler(inputCols=["features", "cluster"], outputCol="augmented_features")
train_augmented = assembler.transform(train_gold).drop("features").withColumnRenamed("augmented_features", "features")
test_augmented = assembler.transform(test_gold).drop("features").withColumnRenamed("augmented_features", "features")

# ΔΙΑΓΡΑΦΗ ΤΟΥ UNDERSAMPLING: Το SMOTE έχει ήδη κάνει τη δουλειά!
# Κρατάμε ολόκληρο το SMOTE dataset.
train_augmented.cache()

print("3. Ορισμός Πλέγματος Παραμέτρων (Grid Search)...")
rf_base = RandomForestClassifier(featuresCol="features", labelCol="stroke", seed=22390225)

# ΔΙΚΑΙΟΛΟΓΗΣΗ: Αφού το SMOTE μας έδωσε χιλιάδες γραμμές, το δέντρο ΜΠΟΡΕΙ 
# και ΠΡΕΠΕΙ να πάει βαθύτερα για να μάθει σωστά, χωρίς να κάνει αμέσως overfit.
paramGrid = (ParamGridBuilder()
             .addGrid(rf_base.maxDepth, [5, 10, 15]) 
             .addGrid(rf_base.numTrees, [100, 200])
             .build())

# Χρησιμοποιούμε PR-AUC, που είναι εξαιρετική μετρική για να αξιολογήσουμε 
# πόσο καλά πέτυχε το SMOTE να ισορροπήσει τα Precision και Recall.
evaluator = BinaryClassificationEvaluator(labelCol="stroke", rawPredictionCol="rawPrediction", metricName="areaUnderPR")

cv = CrossValidator(estimator=rf_base,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator,
                    numFolds=3,
                    seed=22390225)

print("4. Εκτέλεση Cross-Validation στο ΠΛΗΡΕΣ SMOTE Dataset...")
cv_model = cv.fit(train_augmented)
best_rf = cv_model.bestModel

print("\n===========================================================")
print("[ΟΡΘΟΛΟΓΙΚΗ ΕΥΡΕΣΗ SMOTE] Οι βέλτιστες παράμετροι βρέθηκαν:")
print(f" -> maxDepth: {best_rf._java_obj.getMaxDepth()}")
print(f" -> numTrees: {best_rf._java_obj.getNumTrees()}")
print("===========================================================")

print("5. Παραγωγή και αποθήκευση προβλέψεων στο Test Set...")
rf_preds = best_rf.transform(test_augmented)
# Αποθήκευση ως το κεντρικό rf_predictions
rf_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../../data/rf_predictions.parquet")

spark.stop()
print("Ολοκληρώθηκε! Έχεις πλέον ένα RF πλήρως εκπαιδευμένο σε SMOTE δεδομένα.")

1. Εκκίνηση SparkSession...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/09 21:55:58 WARN Utils: Your hostname, cachyos-x8664, resolves to a loopback address: 127.0.1.1; using 192.168.1.5 instead (on interface enp4s0)
26/06/09 21:55:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/09 21:55:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/09 21:55:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/06/09 21:55:59 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


2. Φόρτωση και Feature Augmentation...
3. Ορισμός Πλέγματος Παραμέτρων (Grid Search)...
4. Εκτέλεση Cross-Validation στο ΠΛΗΡΕΣ SMOTE Dataset...


26/06/09 21:56:10 WARN DAGScheduler: Broadcasting large task binary with size 1252.1 KiB
26/06/09 21:56:10 WARN DAGScheduler: Broadcasting large task binary with size 1881.4 KiB
26/06/09 21:56:11 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB
26/06/09 21:56:11 WARN DAGScheduler: Broadcasting large task binary with size 3.4 MiB
26/06/09 21:56:12 WARN DAGScheduler: Broadcasting large task binary with size 2.2 MiB
26/06/09 21:56:13 WARN DAGScheduler: Broadcasting large task binary with size 1506.8 KiB
26/06/09 21:56:13 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/06/09 21:56:14 WARN DAGScheduler: Broadcasting large task binary with size 3.6 MiB
26/06/09 21:56:14 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/06/09 21:56:15 WARN DAGScheduler: Broadcasting large task binary with size 6.8 MiB
26/06/09 21:56:16 WARN DAGScheduler: Broadcasting large task binary with size 4.3 MiB
26/06/09 21:56:17 WARN DAGScheduler: Broadcas


[ΟΡΘΟΛΟΓΙΚΗ ΕΥΡΕΣΗ SMOTE] Οι βέλτιστες παράμετροι βρέθηκαν:
 -> maxDepth: 15
 -> numTrees: 200
5. Παραγωγή και αποθήκευση προβλέψεων στο Test Set...


26/06/09 21:57:33 WARN DAGScheduler: Broadcasting large task binary with size 10.0 MiB


Ολοκληρώθηκε! Έχεις πλέον ένα RF πλήρως εκπαιδευμένο σε SMOTE δεδομένα.
